In [ ]:
# --- path bootstrap: make `src` importable and repo-relative paths resolve ---
# This notebook lives in experiments/notebooks/; run this cell first.
import os, sys
from pathlib import Path

PROJECT_ROOT = Path("/root/Hessian-Approximation-for-Toy-Models")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)  # restore repo root as cwd for relative data/figure paths


# Per-step Hessian approximators on `mlp_08580ee2573a` — 100 steps

Trains the saved-run MLP from init for `MAX_STEPS=100` SGD steps at constant `lr`,
running each per-step Hessian approximator from `HessianComputerRegistry` and
building the trajectory Jacobian as `J = ∏_t (I − lr · H̃_t)`.

EXACT's chain product is the reference. The recurrence is validated once against
`jax.jacobian(train_pure)` on a 20-step run (sanity check).

No damping / pseudo-target sweeping here — only inversion-free chain products.


In [1]:
import json
import os
import tempfile
from pathlib import Path

# Enable f64 BEFORE any JAX import. The env var route is robust to a Jupyter
# kernel that may already have JAX state from another notebook — needed because
# this notebook builds a (n_params, n_params) trajectory Jacobian via 20-100
# matmuls, where f32 unit roundoff would visibly compound (~1e-5 rel err).
os.environ["JAX_ENABLE_X64"] = "1"

import jax

jax.config.update("jax_enable_x64", True)
assert jax.config.read("jax_enable_x64"), (
    "x64 not enabled — restart the kernel and re-run."
)

import jax.numpy as jnp
import optax
from jax.flatten_util import ravel_pytree

from src.config import (
    DatasetEnum,
    HessianApproximationMethod,
    ModelConfig,
    PseudoTargetGenerationStrategy,
)
from src.hessians.collector import CollectorActivationsGradients
from src.hessians.computer.registry import HessianComputerRegistry
from src.hessians.utils.data import ModelContext
from src.utils.data.data import Dataset, load_split_from_disk
from src.utils.loss import cross_entropy_loss
from src.utils.models.registry import ModelRegistry


INFO:2026-06-12 11:35:51,535:jax._src.xla_bridge:810: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory


INFO:2026-06-12 11:35:51,%f:xla_bridge.py:backends:810: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory


In [2]:
PROJECT_ROOT = Path("/root/Hessian-Approximation-for-Toy-Models")
MODEL_DIR = PROJECT_ROOT / "experiments/outputs/models/digits/mlp_08580ee2573a"

with open(MODEL_DIR / "model.json") as f:
    saved = json.load(f)

model_cfg_dict = {k: v for k, v in saved.items() if k != "metadata"}
model_config = ModelConfig.from_dict(model_cfg_dict)
model = ModelRegistry.get_model(model_config, seed=saved["metadata"]["model_seed"])

split_dir = PROJECT_ROOT / saved["metadata"]["dataset"]["split_dir"]
train_ds, test_ds = load_split_from_disk(
    DatasetEnum(saved["metadata"]["dataset"]["name"]),
    split_dir,
)

print(
    f"arch={model_config.architecture.value}, hidden={model_config.hidden_dim}, act={model_config.activation}"
)
print(f"n_train={len(train_ds)}, n_test={len(test_ds)}")
print(f"saved num_parameters={saved['metadata']['num_parameters']}")


arch=mlp, hidden=[16, 16, 16, 16, 16, 16, 16, 16], act=tanh
n_train=3823, n_test=1797
saved num_parameters=2976


In [3]:
# Initial params at the saved run's model_seed. Cache UNRAVEL once so every
# call into the JIT-compiled approximator path sees the same closure (otherwise
# ModelContext would force a recompile per step).
#
# IMPORTANT: Flax's default initializers produce f32 *regardless* of x64 mode,
# so we explicitly cast params (and the train data) to f64. Without this, the
# 20-matmul chain product accumulates ~1e-7 mae and the sanity assert fires.
SEED = saved["metadata"]["model_seed"]
key0 = jax.random.PRNGKey(SEED)
params0 = model.init(key0, jnp.ones((1, model.input_dim)))
params0 = jax.tree.map(lambda x: x.astype(jnp.float64), params0)
flat0, UNRAVEL = ravel_pytree(params0)
N_PARAMS = int(flat0.size)

train_inputs_f64 = train_ds.inputs.astype(jnp.float64)
train_targets = train_ds.targets  # integer labels, leave as-is

LR = 0.03
BATCH_SIZE = 32
MAX_STEPS = 100
N_SANITY_STEPS = 20

assert flat0.dtype == jnp.float64, f"params still {flat0.dtype} after cast"
print(f"n_params={N_PARAMS}, lr={LR}, batch_size={BATCH_SIZE}, max_steps={MAX_STEPS}")
print(f"flat0.dtype={flat0.dtype}, inputs.dtype={train_inputs_f64.dtype}")


n_params=2976, lr=0.03, batch_size=32, max_steps=100
flat0.dtype=float64, inputs.dtype=float64


In [4]:
def make_batches(inputs, targets, batch_size: int, n_steps: int, seed: int):
    """Precompute a deterministic (X_t, Y_t) sequence of length `n_steps`.

    Same shuffle order across every approximator + the sanity check, so all
    runs see identical iterates given identical SGD dynamics. Re-shuffles each
    epoch the way the saved-run training did.
    """
    n = inputs.shape[0]
    n_per_epoch = n // batch_size
    rng = jax.random.PRNGKey(seed)

    Xs, Ys = [], []
    step = 0
    while step < n_steps:
        rng, sk = jax.random.split(rng)
        perm = jax.random.permutation(sk, n)
        Xp = inputs[perm]
        Yp = targets[perm]
        for b in range(n_per_epoch):
            if step >= n_steps:
                break
            sl = slice(b * batch_size, (b + 1) * batch_size)
            Xs.append(Xp[sl])
            Ys.append(Yp[sl])
            step += 1
    return jnp.stack(Xs), jnp.stack(Ys)


In [5]:
def make_hessian_fn(approximator: HessianApproximationMethod, model, loss_fn):
    """Return `(params, X, Y) -> dense (n_params, n_params)` Hessian estimate.

    EXACT / GNH / BLOCK_HESSIAN run only through `ModelContext`. The rest spin
    up a one-batch `CollectorActivationsGradients` pass with EMPIRICAL_FISHER
    targets (k=1, ground-truth labels) and dispatch via the registry. No damping
    or pseudo-target sweeping — we only need H, not its inverse.
    """
    NEEDS_COLLECTOR = approximator not in {
        HessianApproximationMethod.EXACT,
        HessianApproximationMethod.GNH,
        HessianApproximationMethod.BLOCK_HESSIAN,
    }

    def make_ctx(params, X, Y):
        flat, _ = ravel_pytree(params)
        return ModelContext(
            inputs=X,
            params_flat=flat,
            unravel_fn=UNRAVEL,
            model_apply_fn=model.apply,
            loss_fn=loss_fn,
            targets=Y,
            model=model,
        )

    def hessian_fn(params, X, Y):
        model_ctx = make_ctx(params, X, Y)
        if not NEEDS_COLLECTOR:
            comp = HessianComputerRegistry.get_computer(approximator, model_ctx)
            comp.build(base_directory=None)
            return comp.estimate_hessian()

        collector = CollectorActivationsGradients(
            model=model,
            params=params,
            loss_fn=loss_fn,
            pseudo_target_strategy=PseudoTargetGenerationStrategy.EMPIRICAL_FISHER,
            pseudo_target_repetitions=1,
        )
        with tempfile.TemporaryDirectory(prefix="step_collector_") as td:
            collector_data = collector.collect(
                dataset=Dataset(inputs=X, targets=Y),
                save_directory=td,
                try_load=False,
                rng_key=jax.random.PRNGKey(0),
            )
            compute_ctx = HessianComputerRegistry.get_compute_context(
                approximator, collector_data, model_ctx
            )
            comp = HessianComputerRegistry.get_computer(
                approximator, compute_ctx, corr_context=collector_data
            )
            comp.build(base_directory=None)
            return comp.estimate_hessian()

    return hessian_fn


In [6]:
def train_with_approximator(
    *,
    approximator: HessianApproximationMethod,
    params0,
    X_seq,
    Y_seq,
    lr: float,
):
    """SGD over `len(X_seq)` precomputed batches at constant `lr`. At every
    step, materializes `H̃_t` via `approximator` and accumulates the trajectory
    Jacobian `J = ∏_t (I − lr · H̃_t)`.

    Returns dict with `final_params`, `jacobian` ((n_params, n_params)), and
    `losses` ((n_steps,)).
    """
    n_steps = X_seq.shape[0]

    optimizer = optax.sgd(learning_rate=lr, momentum=0.0)
    loss_fn = cross_entropy_loss
    hessian_fn = make_hessian_fn(approximator, model, loss_fn)

    @jax.jit
    def jit_train_step(params, opt_state, X, Y):
        def loss(p):
            return loss_fn(model.apply(p, X), Y)

        loss_value, grads = jax.value_and_grad(loss)(params)
        updates, opt_state = optimizer.update(grads, opt_state)
        return optax.apply_updates(params, updates), opt_state, loss_value

    params = params0
    opt_state = optimizer.init(params)

    eye = jnp.eye(N_PARAMS, dtype=jnp.float64)
    J = eye
    losses = []

    for t in range(n_steps):
        Xb = X_seq[t]
        Yb = Y_seq[t]
        H = hessian_fn(params, Xb, Yb)
        J = (eye - lr * H) @ J
        params, opt_state, loss_value = jit_train_step(params, opt_state, Xb, Yb)
        losses.append(float(loss_value))

    return {
        "final_params": params,
        "jacobian": J,
        "losses": jnp.array(losses),
    }


## Sanity check: chain product matches `jax.jacobian(train)`

`(I − lr · H_t)` recurrence is mathematically exact for vanilla SGD with no
momentum. The original `experiments.ipynb` validated this on a `[32]`-GELU MLP
to ~2e-10 (cell 25). We re-validate here on `[16]*8`-tanh at 20 steps before
trusting it as the reference for the 100-step sweep.


In [7]:
# Pure-JAX SGD so jax.jacobian can differentiate through it.
def _batch_loss_flat(p_flat, X, Y):
    return cross_entropy_loss(model.apply(UNRAVEL(p_flat), X), Y)


def _step_no_aux(p_flat, batch):
    X, Y = batch
    g = jax.grad(_batch_loss_flat)(p_flat, X, Y)
    return p_flat - LR * g, None


def _step_with_hess(p_flat, batch):
    X, Y = batch
    g = jax.grad(_batch_loss_flat)(p_flat, X, Y)
    H = jax.hessian(_batch_loss_flat)(p_flat, X, Y)
    return p_flat - LR * g, H


def train_pure(p_flat, X_seq, Y_seq):
    final, _ = jax.lax.scan(_step_no_aux, p_flat, (X_seq, Y_seq))
    return final


def train_pure_with_hessians(p_flat, X_seq, Y_seq):
    return jax.lax.scan(_step_with_hess, p_flat, (X_seq, Y_seq))


In [8]:
X_seq_sanity, Y_seq_sanity = make_batches(
    train_inputs_f64, train_targets, BATCH_SIZE, N_SANITY_STEPS, seed=SEED
)

# Guard against silent f32 fallback: if x64 didn't take effect, the chain
# product accumulates ~1e-7 mae per 20 matmuls and the assert below fires with
# a misleading "recurrence broken" message. Catch it here instead.
assert flat0.dtype == jnp.float64, (
    f"flat0 is {flat0.dtype}, expected float64. JAX x64 mode is not active in "
    "this kernel — restart the Jupyter kernel and run from cell 1."
)

# Autodiff Jacobian: ∂(final_flat)/∂(initial_flat) over the full 20-step trajectory.
J_auto = jax.jacobian(train_pure)(flat0, X_seq_sanity, Y_seq_sanity)

# Chain product from per-step Hessians collected along the same trajectory.
_, hesses_sanity = train_pure_with_hessians(flat0, X_seq_sanity, Y_seq_sanity)
assert hesses_sanity.dtype == jnp.float64, f"hesses dtype = {hesses_sanity.dtype}"

eye_s = jnp.eye(N_PARAMS, dtype=jnp.float64)
J_chain = eye_s
for H in hesses_sanity:
    J_chain = (eye_s - LR * H) @ J_chain

mae = float(jnp.mean(jnp.abs(J_auto - J_chain)))
rel_frob = float(jnp.linalg.norm(J_auto - J_chain) / jnp.linalg.norm(J_auto))
print(f"sanity check (20 steps):")
print(f"  mean abs |J_auto - J_chain| = {mae:.3e}")
print(f"  ||J_auto - J_chain||_F / ||J_auto||_F = {rel_frob:.3e}")
assert mae < 1e-8, f"chain-product recurrence broken (mae={mae:.3e})"
print("  ok — recurrence valid on this arch.")


sanity check (20 steps):
  mean abs |J_auto - J_chain| = 4.549e-18
  ||J_auto - J_chain||_F / ||J_auto||_F = 6.885e-15
  ok — recurrence valid on this arch.


## Sweep all approximators at 100 steps

Same precomputed batch sequence for every approximator → identical SGD dynamics
across runs (same iterates), so the only thing varying is `H̃_t`.


In [9]:
X_seq, Y_seq = make_batches(
    train_inputs_f64, train_targets, BATCH_SIZE, MAX_STEPS, seed=SEED
)

ALL_APPROXIMATORS = [
    HessianApproximationMethod.EXACT,
    HessianApproximationMethod.KFAC,
    HessianApproximationMethod.EKFAC,
    HessianApproximationMethod.GNH,
    HessianApproximationMethod.FIM,
    HessianApproximationMethod.BLOCK_FIM,
    HessianApproximationMethod.BLOCK_HESSIAN,
    HessianApproximationMethod.SHAMPOO,
    HessianApproximationMethod.ESHAMPOO,
    HessianApproximationMethod.MAC,
    HessianApproximationMethod.EMAC,
    HessianApproximationMethod.IDENTITY,
    HessianApproximationMethod.EIDENTITY,
]

results = {}
for approx in ALL_APPROXIMATORS:
    print(f"=== {approx.value} ===")
    try:
        results[approx.value] = train_with_approximator(
            approximator=approx,
            params0=params0,
            X_seq=X_seq,
            Y_seq=Y_seq,
            lr=LR,
        )
        J = results[approx.value]["jacobian"]
        print(f"  done: ||J||_F = {float(jnp.linalg.norm(J)):.4f}")
    except Exception as e:
        print(f"  FAILED: {type(e).__name__}: {e}")
        results[approx.value] = None


=== exact ===
  done: ||J||_F = 56.6730
=== kfac ===
INFO:2026-06-12 11:36:27,%f:collector.py:_generate_pseudo_targets:202: [PSEUDO-TARGETS] EMPIRICAL_FISHER (ground-truth labels)
INFO:2026-06-12 11:36:27,%f:collector.py:_run_collection_loop:294: [COLLECTION] EMPIRICAL_FISHER: N=32, k=1
INFO:2026-06-12 11:36:28,%f:collector.py:_teardown:335: [TEARDOWN] EMPIRICAL_FISHER: activations (N=32, ...), gradients (N=32, ..., k=1)
INFO:2026-06-12 11:36:28,%f:collector.py:collect:150: Saving collected data to: /tmp/step_collector_yam2ugqk
INFO:2026-06-12 11:36:28,%f:collector.py:save:374: Saved collected data manifest to: /tmp/step_collector_yam2ugqk
INFO:2026-06-12 11:36:30,%f:collector.py:_generate_pseudo_targets:202: [PSEUDO-TARGETS] EMPIRICAL_FISHER (ground-truth labels)
INFO:2026-06-12 11:36:30,%f:collector.py:_run_collection_loop:294: [COLLECTION] EMPIRICAL_FISHER: N=32, k=1
INFO:2026-06-12 11:36:30,%f:collector.py:_teardown:335: [TEARDOWN] EMPIRICAL_FISHER: activations (N=32, ...), gradien

## Comparison vs EXACT chain product

For each approximator, report:

- `pearson(diag(J̃), diag(J_exact))` — diagonal-level alignment.
- `||J̃ − J_exact||_F / ||J_exact||_F` — full-matrix relative error.
- `cosine(J̃, J_exact)` (flattened) — direction alignment.


In [10]:
def pearson_corr(x, y):
    xm = x - jnp.mean(x)
    ym = y - jnp.mean(y)
    eps = jnp.finfo(x.dtype).tiny
    return jnp.sum(xm * ym) / (
        jnp.sqrt(jnp.sum(xm**2)) * jnp.sqrt(jnp.sum(ym**2)) + eps
    )


def cosine(a, b):
    eps = jnp.finfo(a.dtype).tiny
    return jnp.sum(a * b) / (jnp.linalg.norm(a) * jnp.linalg.norm(b) + eps)


J_ref = results["exact"]["jacobian"]
diag_ref = jnp.diag(J_ref)
norm_ref = jnp.linalg.norm(J_ref)

rows = []
for name, run in results.items():
    if run is None:
        rows.append((name, None, None, None))
        continue
    J = run["jacobian"]
    diag_corr = float(pearson_corr(diag_ref, jnp.diag(J)))
    rel_frob = float(jnp.linalg.norm(J - J_ref) / norm_ref)
    cos = float(cosine(J_ref.ravel(), J.ravel()))
    rows.append((name, diag_corr, rel_frob, cos))

print(f"{'method':<16} {'diag(J) corr':>14} {'rel ||·||_F':>14} {'cos(J,J_ref)':>14}")
print("-" * 60)
for name, dc, rf, cs in rows:
    if dc is None:
        print(f"{name:<16} {'FAILED':>14} {'':>14} {'':>14}")
    else:
        print(f"{name:<16} {dc:>14.4f} {rf:>14.4f} {cs:>14.4f}")


method             diag(J) corr    rel ||·||_F   cos(J,J_ref)
------------------------------------------------------------
exact                    1.0000         0.0000         1.0000
kfac                     0.1057         0.2524         0.9677
ekfac                    0.1177         0.2520         0.9678
gnh                      0.1372         0.2544         0.9672
fim                      0.1001         0.2509         0.9681
block_fim                0.1233         0.2521         0.9678
block_hessian            0.3529         0.2550         0.9671
shampoo                  0.1135         0.2521         0.9678
eshampoo                 0.1109         0.2521         0.9678
mac                      0.1513         0.3768         0.9291
emac                     0.2478         0.2519         0.9679
identity                -0.0000         0.9557         0.9684
eidentity                0.3200         0.2510         0.9682


## Init-vector probe: `(J̃ − I) · flat0` per method

Apply each method's trajectory Jacobian to the initial-weight vector itself,
subtract the trivial `I · flat0 = flat0` part, and compare to EXACT's prediction.

`δ_method = (J̃_method − I) · flat0  ∈  ℝ^n_params`

Interpretation: if we'd started training from `flat0 + ε · flat0` (i.e. doubled
the init by a small ε), the linearized prediction of the resulting shift in
final params is `ε · J̃ · flat0`. The `(J̃ − I)` part is the
*curvature-driven* correction over and above "params translate by ε·flat0".
That correction is exactly what each Hessian approximator contributes.


In [11]:
I_n = jnp.eye(N_PARAMS, dtype=jnp.float64)
J_ref = results["exact"]["jacobian"]
ref_image = J_ref @ flat0  # J · w_0
delta_ref = ref_image - flat0  # (J − I) · w_0
norm_w0 = float(jnp.linalg.norm(flat0))
norm_ref_image = float(jnp.linalg.norm(ref_image))
norm_delta_ref = float(jnp.linalg.norm(delta_ref))

print(f"||w_0||                = {norm_w0:.4f}")
print(f"||J_exact · w_0||      = {norm_ref_image:.4f}")
print(f"||(J_exact − I) · w_0|| = {norm_delta_ref:.4f}  (curvature-only correction)")
print()
print(
    f"{'method':<14} {'||J̃·w_0||':>11} {'||(J−J̃)·w_0||':>17} "
    f"{'cos(δ)':>9} {'rel L2':>10}"
)
print("-" * 65)
for name, run in results.items():
    if run is None:
        print(f"{name:<14} FAILED")
        continue
    J = run["jacobian"]
    image = J @ flat0
    delta = image - flat0  # (J̃ − I) · w_0
    abs_diff = float(jnp.linalg.norm(ref_image - image))  # ||J·w_0 − J̃·w_0||
    cos_delta = float(cosine(delta, delta_ref))
    rel_l2 = abs_diff / norm_ref_image  # ||J·w_0 − J̃·w_0|| / ||J·w_0||
    print(
        f"{name:<14} {float(jnp.linalg.norm(image)):>11.4f} "
        f"{abs_diff:>17.4f} {cos_delta:>9.4f} {rel_l2:>10.4f}"
    )


||w_0||                = 11.7675
||J_exact · w_0||      = 12.5360
||(J_exact − I) · w_0|| = 4.0745  (curvature-only correction)

method          ||J̃·w_0||    ||(J−J̃)·w_0||    cos(δ)     rel L2
-----------------------------------------------------------------
exact              12.5360            0.0000    1.0000     0.0000
kfac               11.6557            4.0401    0.1326     0.3223
ekfac              11.6696            4.0609    0.0823     0.3239
gnh                11.5351            4.3438   -0.0462     0.3465
fim                11.5756            4.4801   -0.2290     0.3574
block_fim          11.7142            4.1023   -0.0542     0.3272
block_hessian      11.5992            4.1871   -0.0626     0.3340
shampoo            11.6806            4.0652    0.0694     0.3243
eshampoo           11.6833            4.0779    0.0313     0.3253
mac                10.4009            5.3219    0.0401     0.4245
emac               11.6573            4.0980   -0.0024     0.3269
identity     